In [1]:
%pip install pysr

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os 

import torch
from kan import KAN

import numpy as np
import sympy as sp
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import train_test_split

In [2]:
from pysr import PySRRegressor

/opt/venvs/torch/lib/python3.10/site-packages/juliacall/__init__.py:61: UserWarning: torch was imported before juliacall. This may cause a segfault. To avoid this, import juliacall before importing torch. For updates, see https://github.com/pytorch/pytorch/issues/78829.
  warnings.warn(


Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


In [3]:
dataset_path = 'data_train.csv'
df = pd.read_csv(dataset_path, sep="\t")['y']

dataset_path = 'TSFresh_features.csv'
features = pd.read_csv(dataset_path).drop(['Unnamed: 0'], axis=1)
features

,"value__agg_autocorrelation__f_agg_""var""__maxlag_40",value__kurtosis,value__skewness,value__ratio_beyond_r_sigma__r_1.5,value__lempel_ziv_complexity__bins_5,value__ratio_beyond_r_sigma__r_2,value__lempel_ziv_complexity__bins_3,"value__agg_linear_trend__attr_""intercept""__chunk_len_10__f_agg_""max""",value__ratio_beyond_r_sigma__r_2.5,value__lempel_ziv_complexity__bins_2,...,"value__fft_coefficient__attr_""angle""__coeff_84",value__autocorrelation__lag_2,"value__fft_coefficient__attr_""angle""__coeff_74","value__change_quantiles__f_agg_""mean""__isabs_False__qh_1.0__ql_0.0","value__change_quantiles__f_agg_""mean""__isabs_False__qh_1.0__ql_0.2","value__fft_coefficient__attr_""real""__coeff_86",value__mean_change,"value__fft_coefficient__attr_""real""__coeff_69","value__fft_coefficient__attr_""real""__coeff_57","value__fft_coefficient__attr_""angle""__coeff_89"
0,0.027136,15.344443,3.699521,0.064516,0.093139,0.047706,0.070877,0.270127,0.037256,0.051795,...,-94.464228,0.004890,-38.947448,-0.000455,-0.000455,-2.185269,-0.000455,3.616283,2.598911,-32.824842
1,0.027564,15.606727,3.738659,0.060881,0.087687,0.052703,0.072240,0.266277,0.039527,0.052703,...,-22.594493,-0.003382,-25.665501,-0.000423,-0.000423,-1.212330,-0.000423,0.732944,4.649241,-52.486348
2,0.031556,13.194472,3.485927,0.065425,0.093594,0.054066,0.071786,0.290402,0.043617,0.054066,...,-119.306479,-0.017625,2.377299,-0.000400,-0.000400,2.371623,-0.000400,2.024938,3.540720,-26.102637
3,0.020687,22.265751,4.360557,0.051340,0.074512,0.041799,0.055429,0.193792,0.034984,0.043617,...,-107.525244,-0.002553,-51.556783,-0.000396,-0.000396,2.096422,-0.000396,2.828113,1.745200,-15.233638
4,0.019325,23.690694,4.498539,0.049977,0.071786,0.040891,0.053612,0.180588,0.034075,0.042708,...,-143.535411,-0.001134,-49.950910,-0.000371,-0.000371,-0.585911,-0.000371,4.914692,3.277174,30.077165
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,0.025347,16.140185,3.772223,0.061790,0.086779,0.048160,0.063607,0.252279,0.038619,0.049069,...,59.382482,-0.007052,-0.690027,-0.000380,-0.000380,0.175378,-0.000380,1.067861,1.512779,-5.143551
996,0.023794,19.173884,4.088255,0.056338,0.081781,0.044525,0.059518,0.230399,0.035438,0.046797,...,-35.700821,-0.031018,10.653847,-0.000388,-0.000388,0.235367,-0.000388,1.923052,0.381849,49.253083
997,0.023064,17.571614,3.876513,0.061790,0.085870,0.044071,0.058155,0.236806,0.036347,0.043617,...,-102.261318,0.006745,-8.946237,-0.000455,-0.000455,-2.355916,-0.000455,2.669328,4.102625,-21.091645
998,0.021096,21.580266,4.306552,0.053612,0.077692,0.042708,0.058610,0.203379,0.034984,0.044071,...,-122.227461,-0.004224,-79.947789,-0.000433,-0.000433,1.349140,-0.000433,0.895500,1.457984,-39.627578


In [4]:
X = features
y = df

In [5]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.25, random_state=0)

X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=0)

# Covert data to torch tensor
train_input = torch.tensor(X_train.to_numpy(), dtype=torch.float32)
train_label = torch.tensor(y_train.to_numpy()[:, None], dtype=torch.float32)
val_input = torch.tensor(X_val.to_numpy(), dtype=torch.float32)
val_label = torch.tensor(y_val.to_numpy()[:, None], dtype=torch.float32)
test_input = torch.tensor(X_test.to_numpy(), dtype=torch.float32)
test_label = torch.tensor(y_test.to_numpy()[:, None], dtype=torch.float32)

In [7]:
dataset = {
    'train_input': train_input,
    'train_label': train_label,
    'val_input': val_input,
    'val_label': val_label,
    'test_input': test_input,
    'test_label': test_label
}

In [8]:
# Create KAN
model = KAN(width=[7,3,1], grid=5, k=11)

# Train KAN
results = model.fit(
    train_input,
    train_label,
    val_input,
    val_label,
    loss_fn=torch.nn.MSELoss()
)

checkpoint directory created: ./model
saving model version 0.0


TypeError: only integer tensors of a single element can be converted to an index

In [6]:
X = features.to_numpy()
y = df.to_numpy()

In [13]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=0)

In [14]:
model = PySRRegressor(
    maxsize=25,
    niterations=120, 
    binary_operators=["+", "*"],
    unary_operators=[
        "cos",
        "exp",
        "sin",
        "inv(x) = 1/x",
        # ^ Custom operator (julia syntax)
    ],
    extra_sympy_mappings={"inv": lambda x: 1 / x},
    # ^ Define operator for SymPy as well
    elementwise_loss="loss(prediction, target) = (prediction - target)^2",
    # ^ Custom loss function (julia syntax)
)

In [15]:
model.fit(X_train, y_train)

/opt/venvs/torch/lib/python3.10/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!



Expressions evaluated per second: 0.000e+00
Progress: 0 / 3720 total iterations (0.000%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
───────────────────────────────────────────────────────────────────────────────────────────────────
════════════════════════════════════════════════════════════════════════════════════════════════════
Press 'q' and then <enter> to stop execution early.

Expressions evaluated per second: 6.060e+03
Progress: 72 / 3720 total iterations (1.935%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           5.480e-02  0.000e+00  y = 0.39906
2           5.035e-02  8.478e-02

[ Info: Final population:
[ Info: Results saved to:


,model_selection,'best'
,binary_operators,"['+', '*']"
,unary_operators,"['cos', 'exp', ...]"
,expression_spec,None
,niterations,120
,populations,31
,population_size,27
,max_evals,None
,maxsize,25
,maxdepth,None
,warmup_maxsize_by,None


  - outputs/20251112_040109_TGiAGb/hall_of_fame.csv


In [16]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_pred = model.predict(X_test)

# Avaliar o desempenho
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Resultados no conjunto de teste:")
print(f"MAE:  {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²:   {r2:.4f}")

# Exibir a equação simbólica encontrada
print("\nEquação simbólica encontrada:")
print(model.sympy())

Resultados no conjunto de teste:
MAE:  0.0797
RMSE: 0.0968
R²:   0.7875

Equação simbólica encontrada:
cos(x0*46.959824)


In [12]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_pred = model.predict(X_test)

# Avaliar o desempenho
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Resultados no conjunto de teste:")
print(f"MAE:  {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²:   {r2:.4f}")

# Exibir a equação simbólica encontrada
print("\nEquação simbólica encontrada:")
print(model.sympy())

Resultados no conjunto de teste:
MAE:  0.0812
RMSE: 0.1014
R²:   0.7729

Equação simbólica encontrada:
x1*x82 - 0.957779
